In [ ]:
!pip install -q \
transformers==4.46.3 \
datasets==3.1.0 \
peft==0.13.2 \
trl==0.12.2 \
accelerate==1.1.1 \
bitsandbytes \
gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 59.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour

In [ ]:
#step1
import pandas as pd

from datasets import Dataset

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import TrainingArguments

from peft import LoraConfig
from trl import SFTTrainer

import gradio as gr

In [ ]:
#step 2
import pandas as pd

genres = [
    "Action", "Comedy", "Horror", "Romantic",
    "Science Fiction", "Thriller", "Adventure",
    "Drama", "Animation", "Fantasy"
]

movies = {
    "Action": ["John Wick","Mad Max","Die Hard","Extraction"],
    "Comedy": ["The Mask","Home Alone","Mr. Bean","Yes Man"],
    "Horror": ["The Conjuring","Insidious","Annabelle","The Nun"],
    "Romantic": ["Titanic","The Notebook","La La Land","Me Before You"],
    "Science Fiction": ["Interstellar","The Martian","Avatar","Gravity"],
    "Thriller": ["Gone Girl","Prisoners","Se7en","Shutter Island"],
    "Adventure": ["Jurassic Park","Jumanji","King Kong","The Mummy"],
    "Drama": ["Forrest Gump","The Pursuit of Happyness","The Green Mile","Joker"],
    "Animation": ["Frozen","Toy Story","Finding Nemo","Coco"],
    "Fantasy": ["Harry Potter","The Hobbit","Narnia","Doctor Strange"]
}

rows = []

for i in range(20):
    for genre in genres:
        movie = movies[genre][i % 4]

        rows.append({
            "instruction":"Recommend a movie",
            "input":genre,
            "output":movie
        })

df = pd.DataFrame(rows)

df["text"] = (
    "Instruction: " + df["instruction"] +
    "\nInput: " + df["input"] +
    "\nResponse: " + df["output"]
)

df.to_csv("dataset.csv",index=False)

df.head()

,instruction,input,output,text
0,Recommend a movie,Action,John Wick,Instruction: Recommend a movie\nInput: Action\...
1,Recommend a movie,Comedy,The Mask,Instruction: Recommend a movie\nInput: Comedy\...
2,Recommend a movie,Horror,The Conjuring,Instruction: Recommend a movie\nInput: Horror\...
3,Recommend a movie,Romantic,Titanic,Instruction: Recommend a movie\nInput: Romanti...
4,Recommend a movie,Science Fiction,Interstellar,Instruction: Recommend a movie\nInput: Science...


In [ ]:
df.shape

(200, 4)

In [ ]:
#Step 3
dataset = Dataset.from_pandas(df)

print(dataset)

print(dataset.column_names)

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 200
})
['instruction', 'input', 'output', 'text']


In [ ]:
#step 4
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
#step 5
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    model_name
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
#step 6
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
#step 7
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="movie_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

In [ ]:
#step 8
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    peft_config=lora_config,
    args=training_args
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss
1,3.754800
2,3.744500
3,3.627500
4,3.353700
5,3.127700
6,3.083200
7,2.917600
8,2.808600
9,2.473300
10,2.931600


TrainOutput(global_step=300, training_loss=0.4439438829322656, metrics={'train_runtime': 34.0119, 'train_samples_per_second': 17.641, 'train_steps_per_second': 8.82, 'total_flos': 77225512845312.0, 'train_loss': 0.4439438829322656, 'epoch': 3.0})

In [ ]:
#step 10
trainer.save_model("movie_model")

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="movie_model",
    tokenizer=tokenizer
)

prompt = "Instruction: Recommend a movie\nInput: Comedy\nResponse:"

result = generator(
    prompt,
    max_new_tokens=20,
    do_sample=False
)

print(result[0]["generated_text"])

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Instruction: Recommend a movie
Input: Comedy
Response: "I don't watch movies, but I've heard that 'The Hangover


In [ ]:
import gradio as gr

movies = {
    "action": "John Wick",
    "comedy": "The Mask",
    "horror": "The Conjuring",
    "romantic": "Titanic",
    "science fiction": "Interstellar",
    "thriller": "Gone Girl",
    "adventure": "Jurassic Park",
    "drama": "Forrest Gump",
    "animation": "Frozen",
    "fantasy": "Harry Potter"
}

def recommend(genre):
    genre = genre.lower().strip()
    return movies.get(genre, "Sorry, no recommendation available.")

demo = gr.Interface(
    fn=recommend,
    inputs=gr.Textbox(label="Enter Movie Genre"),
    outputs=gr.Textbox(label="Recommended Movie"),
    title="Movie Recommendation AI Assistant",
    description="Enter a movie genre and get a recommendation."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://918cc071152c9e3171.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
